In [ ]:
import os
import requests
import msal
from datetime import datetime, timezone

TENANT_ID = os.environ["AZURE_TENANT_ID"]
CLIENT_ID = os.environ["AZURE_CLIENT_ID"]
DATAVERSE_URL = os.environ["DATAVERSE_URL"]

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = [f"{DATAVERSE_URL}/.default"]

TABLE_SET_NAME = "REPLACE_WITH_TABLE_SET_NAME"
PRIMARY_KEY = "REPLACE_WITH_PRIMARY_KEY_LOGICAL_NAME"

PREDICTION_COLUMN = "REPLACE_WITH_PREDICTION_COLUMN_LOGICAL_NAME"
SCORE_COLUMN = "REPLACE_WITH_SCORE_COLUMN_LOGICAL_NAME"
SCORED_ON_COLUMN = "REPLACE_WITH_SCORED_ON_COLUMN_LOGICAL_NAME"
STATUS_COLUMN = "REPLACE_WITH_STATUS_COLUMN_LOGICAL_NAME"

def get_access_token():
    app = msal.PublicClientApplication(
        client_id=CLIENT_ID,
        authority=AUTHORITY
    )

    result = app.acquire_token_interactive(scopes=SCOPE)

    if "access_token" not in result:
        raise RuntimeError(f"Authentication failed: {result}")

    return result["access_token"]


def get_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
        "OData-MaxVersion": "4.0",
        "OData-Version": "4.0"
    }


def fetch_records_to_score(token):
    headers = get_headers(token)

    # Replace this filter with your real status/flag column.
    url = (
        f"{DATAVERSE_URL}/api/data/v9.2/{TABLE_SET_NAME}"
        f"?$top=50"
    )

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.json().get("value", [])


def run_local_model(record):
    """
    Replace this with your ONNX Runtime or joblib inference code.
    """
    prediction = "ExamplePrediction"
    score = 0.95
    return prediction, score


def update_record(token, row_id, prediction, score):
    headers = get_headers(token)

    url = f"{DATAVERSE_URL}/api/data/v9.2/{TABLE_SET_NAME}({row_id})"

    payload = {
        PREDICTION_COLUMN: prediction,
        SCORE_COLUMN: score,
        SCORED_ON_COLUMN: datetime.now(timezone.utc).isoformat()
    }

    response = requests.patch(url, headers=headers, json=payload)
    response.raise_for_status()


def main():
    token = get_access_token()

    records = fetch_records_to_score(token)

    for record in records:
        row_id = record[PRIMARY_KEY]

        prediction, score = run_local_model(record)

        update_record(
            token=token,
            row_id=row_id,
            prediction=prediction,
            score=score
        )

        print(f"Updated row {row_id}: {prediction}, {score}")


if __name__ == "__main__":
    main()